# 05 · Deploy the demo to Hugging Face Spaces
Builds lightweight demo artifacts (1% coreset banks, thresholds, gallery) and pushes the app straight from Colab to your Space.
You need a Hugging Face account and a **write** token (huggingface.co → Settings → Access Tokens). Add it as a Colab secret named `HF_TOKEN` (key icon in the left bar), or paste it when prompted.
Runtime: T4 GPU makes the export faster, CPU also works.

### Setup (run every session)
Mounts Google Drive (where data, caches and results live), clones or updates the repo, installs requirements.
**Edit `REPO_URL` once** to point at your GitHub repo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/<your-username>/defectlens.git"   # <-- change this
import os
if not os.path.exists('/content/defectlens'):
    !git clone -q {REPO_URL} /content/defectlens
else:
    !git -C /content/defectlens pull -q
%cd /content/defectlens
!pip install -q -r requirements.txt
!mkdir -p /content/drive/MyDrive/defectlens/work

In [ ]:
!python scripts/export_demo.py

In [ ]:
import os, shutil, getpass
from huggingface_hub import HfApi
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = getpass.getpass("HF write token: ")

SPACE_ID = "<your-hf-username>/defectlens"   # <-- change this
api = HfApi(token=token)
api.create_repo(SPACE_ID, repo_type="space", space_sdk="docker", exist_ok=True)

stage = "/content/space_stage"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree("defectlens", f"{stage}/defectlens")
shutil.copytree("configs", f"{stage}/configs")
os.makedirs(f"{stage}/app")
shutil.copy("app/streamlit_app.py", f"{stage}/app/streamlit_app.py")
shutil.copytree("/content/drive/MyDrive/defectlens/work/demo_artifacts", f"{stage}/app/artifacts")
shutil.copy("deploy/Dockerfile", f"{stage}/Dockerfile")
shutil.copy("deploy/requirements-space.txt", f"{stage}/requirements-space.txt")
shutil.copy("deploy/SPACE_README.md", f"{stage}/README.md")

api.upload_folder(folder_path=stage, repo_id=SPACE_ID, repo_type="space",
                  commit_message="Deploy DefectLens demo")
print(f"Deployed: https://huggingface.co/spaces/{SPACE_ID}  (first build takes a few minutes)")